# Telco Customer Churn Prediction: MLOps Lifecycle

Welcome to the MLOps Lab! This project is a showcase of a production-quality,
end-to-end Machine Learning Operations lifecycle for **batch churn scoring** —
the canonical "score the whole population on a schedule" deployment pattern.

We implement the full CD4ML (Continuous Delivery for ML) lifecycle: every
active customer in a CRM snapshot export is scored each cycle by a churn
model, producing a ranked risk list for the retention team.

## Why Batch Transform + Feast Offline?

Churn signals are slow-moving subscription attributes refreshed at billing
cycles — nothing is scored per-request, so there is no synchronous endpoint
to keep warm. **SageMaker Batch Transform** spins up a serving container,
streams the population through the model, writes predictions, and exits;
**Feast's offline store** gives us point-in-time-correct feature retrieval
for both training and scoring from a single feature definition — no
training-serving skew.

## Architecture

```mermaid
flowchart LR
    A[CRM snapshot CSV] --> B[Clean & feature-build]
    B --> C[Feast offline store<br/>DuckDB + parquet]
    C --> D[Historical features<br/>training]
    C --> E[Historical features<br/>full population]
    D --> F[XGBoost<br/>SageMaker training job]
    F --> G[Holdout evaluation]
    E --> H[SageMaker Batch Transform]
    G --> H
    H --> I[Scored risk list]
    I -. drift review .-> B
```

Everything runs fully offline in Docker: **moto** emulates S3/STS, SageMaker
local mode runs the jobs in local containers, and the only service this
project's `docker-compose.yml` declares is that moto — a batch store needs
no always-on infrastructure.

## Lifecycle stages

| Stage | Page | Contents |
|---|---|---|
| Problem framing | Problem Framing | Business objective, error trade-offs, task formulation |
| Data collection | Data Collection | The IBM Telco churn CRM snapshot contract |
| EDA | Exploratory Data Analysis | Distributions, churn drivers, correlations |
| Preprocessing | Data Preprocessing | Cleaning, encoding, output contracts |
| Feature engineering | Feature Engineering | Feast repo, offline store, historical retrieval |
| Training | Model Training | XGBoost via a SageMaker training job on moto S3 |
| Tuning | Hyperparameter Tuning | A small local sweep + per-job metrics.json |
| Evaluation | Model Evaluation | Transform-on-holdout, confusion matrix, AUC |
| Deployment | Deploy | Batch Transform scoring the full population |
| Monitoring | Monitoring | Feature drift review + stack teardown |

## How to run

Run the notebooks inside the repo's JupyterLab container
(`docker compose up jupyterlab` at the repo root), where `docker`, `feast`
and `aws` are on the kernel PATH and the project's `docker-compose.yml`
supplies moto. Build the images once with `make build-images`.
